In [ ]:
!nvidia-smi

In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install --upgrade accelerate
!pip uninstall -y transforers accelerate
!pip install transformers accelerate

In [ ]:
!pip install evaluate

In [ ]:
from transformers import pipeline, set_seed
from datasets import load_dataset , load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
from datasets import load_dataset
import evaluate

from transformers import AutoModelForSeq2SeqLM , AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

In [ ]:
from transformers import AutoModelForSeq2SeqLM , AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

In [ ]:
# Load from Hugging Face Hub
dataset_samsum = load_dataset("knkarthick/samsum")

# Save the dataset to disk
dataset_samsum.save_to_disk("samsum_dataset")


In [ ]:
dataset_samsum = load_from_disk("samsum_dataset")
print(dataset_samsum)


In [ ]:
split_lengths = [len(dataset_samsum[split]) for split in dataset_samsum]

print(f"Split lengths:{split_lengths}")
print(f"Features: {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])

print("\nSummary")

print(dataset_samsum["test"][1]["summary"])

In [ ]:
def convert_examples_to_features(example_batch):
  input_encodings = tokenizer(example_batch['dialogue'] , max_length = 1024 ,truncation=True)
  target_encodings = tokenizer(
      text_target=example_batch["summary"],
      max_length=128,
      truncation=True
  )
  return{
      'input_ids' : input_encodings['input_ids'],
      'attention_mask': input_encodings['attention_mask'],
      'labels': target_encodings['input_ids']
  }

In [ ]:
dataset_samsum_pt = dataset_samsum.map(convert_examples_to_features , batched = True)

In [ ]:
dataset_samsum_pt["train"]

In [ ]:
#Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collector = DataCollatorForSeq2Seq(tokenizer , model = model_pegasus)


In [ ]:
from transformers import TrainingArguments

trainer_args = TrainingArguments(
    output_dir="pegasus-samsum",
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",   # v5 syntax
    eval_steps=500,
    save_steps=1_000_000,
    gradient_accumulation_steps=16
)

In [ ]:
from transformers import TrainingArguments , Trainer

# trainer = Trainer(
#     model=model_pegasus,
#     args=trainer_args,
#     data_collator=seq2seq_data_collator,
#     train_dataset=dataset_samsum_pt["train"],
#     eval_dataset=dataset_samsum_pt["validation"]
# )

from transformers import DataCollatorForSeq2Seq

# Create a data collator for seq2seq tasks
seq2seq_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model_pegasus
)

# Now you can safely pass it into Trainer
trainer = Trainer(
    model=model_pegasus,
    args=trainer_args,
    data_collator=seq2seq_data_collator,
    train_dataset=dataset_samsum_pt["train"],
    eval_dataset=dataset_samsum_pt["validation"]
)



In [ ]:
trainer.train()

In [ ]:
#Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """Split the dataset into smaller batches we can process simultaneously.
    Yield successive bacth-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                                batch_size=16, device=device,
                                column_text="dialogue",
                                column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)
    ):
        inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                           padding="max_length", return_tensors="pt")

        summaries = model.generate(
            input_ids=inputs["input_ids"].to(device),
            attention_mask=inputs["attention_mask"].to(device),
            length_penalty=0.8,
            num_beams=8,
            max_length=128
        )
        '''parameter for length penalty ensures that the model does not generate sequences'''

        # Decode generated summaries
        decoded_summaries = tokenizer.batch_decode(summaries, skip_special_tokens=True)
        # decoded_summaries= [d.replace(""," ") for d in decoded_summaries]

        # Add batch to metric
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Compute and return ROUGE scores
    score = metric.compute()
    return score

In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = evaluate.load("rouge")

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_samsum_pt["test"], rouge_metric, trainer.model, tokenizer, batch_size=2 , column_text="dialogue" , column_summary="summary"
)
rouge_dict = {rn: score[rn] for rn in rouge_names}
pd.DataFrame(rouge_dict, index=["pegasus"])


In [ ]:
# Save model
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
#Save tokenizer
tokenizer.save_pretrained("tokenizer")

In [ ]:
# Load

tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
# Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams": 8, "max_length": 128}

sample_text = dataset_samsum_pt["test"][0]["dialogue"]
reference = dataset_samsum_pt["test"][0]["summary"]

# Encode the input dialogue and move to GPU if available
inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=1024
).to(trainer.model.device)   # match model device

# Generate summary
summary_ids = trainer.model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    **gen_kwargs
)

# Decode the generated summary
model_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference)

print("\nModel Summary:")
print(model_summary)


